# 03 · Messdateien lokal prüfen

**Aufgabe:** CSV-Dateien sichten, Zahlen und Zeilen prüfen und eine lokale Importvorschau erzeugen.

Frischer Kernel im Repository, dann **Run All**. Funktioniert offline und bei leerem
Messdatenordner. Originaldateien bleiben unverändert; die Vorschau liegt ausschließlich
unter der ignorierten Ablage `data/measurements/`. Es erfolgt kein Upload.
Das aktuelle Setup wird nur als möglicher Kontext angezeigt, keiner CSV automatisch zugeordnet.

In [ ]:
from pathlib import Path
from gasi_setup import inspect_setup, measurement_context

repo_root = Path.cwd()
measurement_dir = repo_root / "data" / "measurements"
files = sorted(measurement_dir.glob("*.csv"))
print(f"CSV-Dateien: {len(files)}")
for path in files[:30]:
    print(f"- {path.name} ({path.stat().st_size:,} Bytes)")
if len(files) > 30:
    print(f"Weitere {len(files) - 30} Dateien werden ebenfalls geprüft.")
try:
    setup_context = measurement_context(inspect_setup(repo_root / "config/gasi_setup.json"))
except ValueError:
    setup_context = None
    print("Setup-Konfiguration noch unvollständig/ungültig; CSV-Prüfung läuft unabhängig weiter.")

## Validierung und lokale Vorschau

Zu kurze/lange Zeilen, ungültige Zahlen, `NaN` und Unendlichkeiten werden abgewiesen.
Leere optionale Werte bleiben von Null unterscheidbar. Die Notebook-Ausgabe enthält
keine Messpunktwerte. Datei und Messlauf sind verschiedene Einheiten.

In [ ]:
import json
from pathlib import Path

from gasi_local_measurements import inspect_measurements

repo_root = Path.cwd()
measurement_dir = repo_root / "data" / "measurements"
preview_path = measurement_dir / "_local_openbis_import_preview.json"
summary = inspect_measurements(measurement_dir, preview_path, setup_context=setup_context)

if summary["files_checked"] == 0:
    print(summary["message"])
    print("CSV-Prüfung und Importvorschau wurden übersprungen.")
else:
    print("Lokale CSV-Validierung:")
    print(
        f"- Dateien/Zeilen: {summary['files_checked']} / {summary['rows_checked']}; "
        f"Schema-Varianten/Spalten: {summary['schema_variants']} / {summary['column_count']}"
    )
    print(
        f"- Parse-/Zahlen-/Zeilenfehler: {summary['parse_errors']}/"
        f"{summary['numeric_errors']}/{summary['row_shape_errors']}"
    )
    print(
        f"- Mögliche lokale Uhr-Rückstellungen: {len(summary['ambiguous_clock_rollbacks'])}; "
        "Zeitzone fehlt im CSV"
    )
    print(f"- Häufige Abtastintervalle (s): {summary['sampling_intervals_seconds']}")
    print(
        f"- Projektlabels/Gasprofile/Komponentengase: {len(summary['project_names'])}/"
        f"{len(summary['gas_profiles'])}/{summary['component_ids']}"
    )
    print(
        f"- Kalibrierungseinträge: {summary['calibration_entries']}; "
        f"JSON-Payloads leer: {summary['calibration_payloads_empty']}"
    )
    print(f"- Beispiel-Datei: {summary['example_file'].name}")
    print(f"- Lokale Vorschau: {summary['preview_path'].relative_to(repo_root).as_posix()}")
    print("- Beispiel-Messwerte werden nicht in Notebook-Ausgaben angezeigt.")

    preview = json.loads(summary["preview_path"].read_text(encoding="utf-8"))
    print("\nopenBIS-Importvorschau (nur Struktur):")
    print(json.dumps(
        {
            "object_type": preview["openbis_path_proposal"]["object_type"],
            "property_sources": list(preview["object_properties"]),
            "relationship_status": [item["status"] for item in preview["relationships"]],
            "datasets": preview["datasets"],
            "unresolved": preview["unresolved"],
            "field_semantics": preview["field_semantics"],
            "locally_saved_point_examples": summary["example_points"],
        },
        ensure_ascii=False,
        indent=2,
    ))